In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/pre_cell_3.pickle

trying: ['lineitem']


me:  1
trying: ['pd']
me:  0
trying: ['load_customer']
me:  3
trying: ['customer']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['orders']


me:  2


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Filter and drop unused columns early
define_date = '1995-03-04'

fcustomer = customer.query("C_MKTSEGMENT == 'HOUSEHOLD'")[['C_CUSTKEY']]
forders  = orders.query(f"O_ORDERDATE < '{define_date}'")[['O_ORDERKEY','O_CUSTKEY','O_ORDERDATE','O_SHIPPRIORITY']]
flineitem = lineitem.query(f"L_SHIPDATE > '{define_date}'")[['L_ORDERKEY','L_EXTENDEDPRICE','L_DISCOUNT']]

# Join on the correct key names
jn2 = (
    fcustomer
      .merge(forders, left_on='C_CUSTKEY', right_on='O_CUSTKEY')
      .merge(flineitem, left_on='O_ORDERKEY', right_on='L_ORDERKEY')
)

# Compute TMP, aggregate and sort
total = (
    jn2
      .assign(TMP = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT))
      .groupby(
          ['L_ORDERKEY','O_ORDERDATE','O_SHIPPRIORITY'],
          as_index=False,
          sort=False
      )
      .agg({'TMP':'sum'})
      .sort_values('TMP', ascending=False)
)

# Final projection
res = total[['L_ORDERKEY','TMP','O_ORDERDATE','O_SHIPPRIORITY']]


CPU times: user 40 s, sys: 7.25 s, total: 47.3 s
Wall time: 47.3 s


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle

migration speed (bps): 786197407.1736864
---------------------------
variables to migrate:
load_lineitem 144
total 48
DATA_ROOT 80
pd 72
orders 48
STORAGE_OPTS 64
load_orders 144
forders 48
define_date 59
tpch_parent 54
load_customer 144
jn2 48
fcustomer 48
res 48
customer 48
lineitem 48
flineitem 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables ['DATA_ROOT', 'STORAGE_OPTS']
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables ['customer', 'lineitem', 'orders']
Active variables []
Intermediate variables ['customer', 'flineitem', 'jn2', 'res', 'define_date', 'orders', 'fcustomer', 'forders', 'lineitem', 'total']
Future variables []
Modified dataframes
  lineitem
    Input columns: set()
    Changed columns: {'L_EXTENDEDPRICE', 'L_SHIPDATE', 'L_RECEIPTDATE', 'L_SHIPMODE', 'L_LINESTATUS', 'L_SHIPINSTRUCT', 'L_SUPPKEY', 'L_ORDERKEY', 'L_TAX', 'L_RETURNFLAG', 'L

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q03/opt_cell_exec_info_3_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [8]:
opt_output = Out.get(4)

In [9]:
res_opt = res
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['date']
me:  4
trying: ['forders']
me:  4
trying: ['orig_output']
me:  5
trying: ['csel']
me:  4
trying: ['STORAGE_OPTS']
me:  0
trying: ['jn2']
me:  4
trying: ['osel']
me:  4
trying: ['res']
me:  4
trying: ['fcustomer']
me:  4
trying: ['load_orders']
me:  2
trying: ['DATA_ROOT']
me:  0
trying: ['orders']


me:  2
trying: ['lineitem_filtered']


me:  4
trying: ['total']
me:  4
trying: ['orders_filtered']
me:  4
trying: ['customer']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['pd']
me:  0
trying: ['lineitem']


me:  1
trying: ['load_customer']
me:  3
trying: ['lsel']
me:  4
trying: ['jn1']
me:  4
trying: ['customer_filtered']
me:  4
trying: ['flineitem']
me:  4
trying: ['tpch_parent']
me:  0


Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable customer
Declaring variable load_customer
Declaring variable date
Declaring variable forders
Declaring variable csel
Declaring variable jn2
Declaring variable osel
Declaring variable res
Declaring variable fcustomer
Declaring variable lineitem_filtered
Declaring variable total
Declaring variable orders_filtered
Declaring variable lsel
Declaring variable jn1
Declaring variable customer_filtered
Declaring variable flineitem
Declaring variable orig_output
